## 01- Transformer Classifier Baseline
In this notebook, we will do a baseline using a mix of classicial and neural method (Transformer-based sentence embeddings and neural classifier)

In [1]:
!pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score


C:\Users\Minh UBC\miniconda3\Lib\site-packages\transformers\utils\generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [3]:
train_fname = "../data/splits/train.csv"
dev_fname = "../data/splits/dev.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 132
Dev df = 34


In [4]:
train_df.head()

,text,target
0,Reaching for a cookie chair possibly moving to...,CN
1,As a picture little girl and a little boy stan...,AD
2,Tell me everything that you see happening. Eve...,CN
3,I think what I want you to do is take a look a...,CN
4,Tell me all the action that you can say on wha...,AD


In [5]:
dev_df.head()

,text,target
0,"Everything you see going on in this picture, g...",CN
1,"Now you just look at the picture. Okay, this i...",CN
2,Tell me everything that you see going on in th...,AD
3,Girls and boys getting her one that stores sto...,AD
4,"All right, we have a few more tasks to go thro...",CN


### Pre-processing
We will lowercase the text for better efficiency

In [6]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

### Apply Sentence Transformer

We can use Sentence Transformer to convert text to a fixed-length embedding vector

In [7]:
vectorizer = SentenceTransformer("sentence-t5-large") # works best

C:\Users\Minh UBC\miniconda3\Lib\site-packages\transformers\utils\generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
C:\Users\Minh UBC\miniconda3\Lib\site-packages\transformers\utils\generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [8]:
corpus = train_df["text_lower"]
print("Creating text embeddings...")
X_train = vectorizer.encode(corpus)
print("Completed")

Creating text embeddings...
Completed


In [9]:
print(X_train.shape)

(132, 768)


### Train Neural Classifier

We can now train a neural logistic regression classifier using processed X_train

In [10]:
y_train = train_df["target"]

print("Training neural classifier...")
hidden_size = 500
clf = MLPClassifier(solver="lbfgs",
                    alpha=1e-5,
                    hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
                    # activation="relu",
                    # shuffle=True,
                    # learning_rate="constant",
                    learning_rate_init=0.0001,
                    max_iter=500, 
                    random_state=42
                   )
clf.fit(X_train, y_train)
print("Completed")

Training neural classifier...
Completed


In [11]:
# getting score on train set
train_score = clf.score(X_train, y_train)
print(f"Train accuracy = {train_score*100:.2f} %")

Train accuracy = 100.00 %


### Get accuracy using cross validation

In [12]:
print("Performing cross-validation...")
scores = cross_val_score(clf, X_train, y_train, cv=10)
print("Completed")
print(scores)

Performing cross-validation...
Completed
[0.78571429 0.64285714 0.69230769 0.76923077 0.76923077 0.84615385
 0.84615385 0.69230769 0.69230769 0.53846154]


In [13]:
print("Average =", np.average(scores))

Average = 0.7274725274725274


In [14]:
# get precision / recall / f1
y_pred = clf.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

          AD       1.00      1.00      1.00        66
          CN       1.00      1.00      1.00        66

    accuracy                           1.00       132
   macro avg       1.00      1.00      1.00       132
weighted avg       1.00      1.00      1.00       132



### Now we can evaluate on dev set

In [15]:
X_dev = vectorizer.encode(dev_df["text_lower"])
y_dev = dev_df["target"]
print("X_dev shape =", X_dev.shape)
print("y_dev len =", len(y_dev))

X_dev shape = (34, 768)
y_dev len = 34


In [16]:
dev_score = clf.score(X_dev, y_dev)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 82.35 %


In [17]:
y_pred = clf.predict(X_dev)
print(classification_report(y_dev, y_pred))

              precision    recall  f1-score   support

          AD       0.83      0.90      0.86        21
          CN       0.82      0.69      0.75        13

    accuracy                           0.82        34
   macro avg       0.82      0.80      0.81        34
weighted avg       0.82      0.82      0.82        34

